# jupygate

> Websocket gateway for Jupyter kernels

In [ ]:
#| hide
from jupygate.core import *

`jupygate` starts, stops, and restarts Jupyter kernels through a small HTTP API. It multiplexes each kernel's zmq channels over one websocket connection per client. It implements the standard Jupyter kernel REST API and legacy Jupyter websocket message protocol. Existing Jupyter-compatible clients work unchanged. It has no files API, kernelspec lookup, or HTML interface. It is a small, readable alternative to running `jupyter_server` or `kernel_gateway` for the one job of hosting kernels.

The gateway manages kernel connections and client messages as follows:

- Each kernel's websocket clients share one persistent set of zmq channels owned by the gateway. This avoids `jupyter_server`'s per-connection readiness checks, buffering handoffs, and associated reconnect races.
- Kernels implementing JEP 65, including `ipymini` and recent `ipykernel` versions, send `iopub_welcome` when an iopub subscription is ready. The gateway then completes a `kernel_info` round trip. Kernels without welcome support use `jupyter_client`-style retries. Both paths use the same readiness check with a timeout.
- Each client uses its own session id. The gateway routes shell, control, and stdin messages using `parent_header.session`. It broadcasts iopub messages to all connected clients. The kernel's HMAC key stays in the gateway. Clients authenticate with a token at the HTTP/websocket layer.
- The gateway limits queued iopub output for each connected client. It drops excess output when the queue fills and retains `status` messages. Clients can still track the kernel's busy/idle state during an output flood.

The source, documentation, and tests are together in one nbdev notebook, [`core`](00_core.ipynb). It covers both websocket encodings, kernel processes, zmq channels and readiness checks, message routing and broadcasts, and the HTTP/websocket server, in dependency order.

## Install

```sh
pip install jupygate
```

Install a Jupyter kernel too. The examples use [ipymini](https://github.com/AnswerDotAI/ipymini).

## Use

Start a local gateway:

```sh
jupygate --port 8787
```

Create and manage kernels through HTTP:

```sh
curl -X POST localhost:8787/api/kernels          # -> {"id": "...", ...}
curl localhost:8787/api/kernels                  # list kernels
curl -X POST localhost:8787/api/kernels/ID/interrupt
curl -X DELETE localhost:8787/api/kernels/ID
```

Replace `ID` with the kernel id returned by the API. Connect a websocket to `/api/kernels/ID/channels?session_id=...` to exchange messages with that kernel. Send Jupyter message dicts with a `channel` key. The wire-formats section of [`core`](00_core.ipynb) describes the encodings.

Kernel creation accepts launch parameters in the POST body:

- `argv`: the command to run, with `{connection_file}` as a placeholder for the kernel connection file.
- `env`: the process environment.
- `cwd`: the working directory.
- `username`: the user account to run the kernel under, using `sudo` or a configured helper such as `gosu`.

Kernel creation can run arbitrary commands on the server. Set an authentication token before exposing the gateway to other machines:

```sh
jupygate --port 8787 --token S
```

Replace `S` with your token. Clients must supply it in an `Authorization: token S` header or a `token=S` query parameter on HTTP and websocket requests.

To run the gateway from Python:

```python
from jupygate.core import create_app, serve
serve(create_app(auth_token='S'), port=8787)
```

The CLI creates `~/.local/state/jupygate/reload/r.py` at startup. Touch this file after upgrading `jupygate` or a kernel package to restart the gateway with the new code. Restarting stops all kernels. Pass `--reload` to also restart when the package source changes during development. The Python `serve()` function does not watch for changes.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()